# API Fetch of Planned Green Spaces Data (Espaces verts et assimilés par projet) Data
---

**Source:** https://opendata.paris.fr/explore/?disjunctive.theme&disjunctive.publisher&disjunctive.keyword&disjunctive.modified&disjunctive.features&sort=modified

**Endpoint used:** `espaces_verts_supplementaires`  

**Last updated:** April 10, 2026

**Description:**

This indicator, which feeds into the indicator of green public space, is linked to the objective of creating 30 hectares of additional green spaces during the term of office. The areas are expressed in hectares and have been counted since August 2020.

## 1. API Response Check

In [1]:
import requests
import pandas as pd
import json
from tqdm.notebook import tqdm


url  = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets"
dataset = "espaces_verts_supplementaires"


# Create the endpoint URL using .format()
endpoint = "{}/{}/records".format(url, "espaces_verts_supplementaires")

# Perform a GET request
response = requests.get(endpoint)

# Check the status code (200 = successful)
print("Status code:", response.status_code)

# Check the keys of the response
data = response.json()
print(data.keys())

Status code: 200
dict_keys(['total_count', 'results'])


## 2. Fetch data and create dataframe

In [2]:
all_results = []
limit = 100
offset = 0

# Loop to fetch all records using pagination
while True:
    endpoint = "{}/{}/records?rows={}&start={}".format(url, dataset, limit, offset)
    response = requests.get(endpoint)

    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code} - {response.json().get('message', 'Unknown error')}")
        break

    data = response.json()
    current_results = data.get('results', [])
    all_results.extend(current_results)

    if len(current_results) < limit:
        # No more data to fetch, break the loop
        break

    offset += limit

# Update the 'data' variable with the combined results for consistency with subsequent cells
data = {'results': all_results, 'total_count': len(all_results)}
print(f"Successfully fetched {len(all_results)} records.")

Successfully fetched 73 records.


In [3]:
df_green_spaces = pd.DataFrame(all_results)


# Check length of results
if len(data["results"]) == len(all_results):
    print("Length of results is equal to records available.")
else: print("Records fetched do not match records available.")

Length of results is equal to records available.


## 3. Data Preprocessing & Exploration

#### Assess variables and their types

In [4]:
# First look at dataframe and variable types

display(df_green_spaces.head(10))
print("\n\n")
display(df_green_spaces.info())

,nom_projet,arrond,secteur_adm,date_fin_t,ope_type,ind_ev_suppl
0,Transformation de la cour de l’espace Gabriel ...,12,12,2025,Nouvelle surface d’espaces verts,830.0
1,Végétalisation du Centre Sportif Elisabeth,14,14,2023,Nouvelle surface d’espaces verts,243.0
2,Renaturation du square Antoine Blondin,20,20,2022,Nouvelle surface d’espaces verts,949.0
3,Renaturation et extension du square des Périch...,15,15,2021,Nouvelle surface d’espaces verts,860.0
4,Ouverture au public PC18 - Bvd Ornano / rue de...,18,18,2024,Nouvelle surface d’espaces verts,13440.0
5,Renaturation et extension du Jardin de la Nouv...,8,8,2024,Nouvelle surface d’espaces verts,2850.0
6,Parking Parc Floral,12,12,2023,Nouvelle surface d’espaces verts,0.0
7,AAP Parisculteurs AUTRES - Tunnel Friant,14,14,2021,Nouvelle surface d’espaces verts,0.0
8,Renaturation du jardin Curial,19,19,2025,Nouvelle surface d’espaces verts,515.0
9,Requalification de l'avenue du Général Eisenhower,8,8,2024,Nouvelle surface d’espaces verts,527.0





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   nom_projet    73 non-null     object 
 1   arrond        73 non-null     object 
 2   secteur_adm   73 non-null     object 
 3   date_fin_t    73 non-null     object 
 4   ope_type      73 non-null     object 
 5   ind_ev_suppl  73 non-null     float64
dtypes: float64(1), object(5)
memory usage: 3.6+ KB


None

In [5]:
# Convert value types
df_green_spaces['arrond'] = df_green_spaces['arrond'].astype(int).sort_values()
df_green_spaces['date_fin_t'] = df_green_spaces['date_fin_t'].astype(int).sort_values()

In [6]:
# Variable type classification
# Classifies each column by dtype and number of unique values.
# Thresholds defined based on the actual distribution of unique values in this dataset.

categories = []

for col in df_green_spaces.columns:
    series = df_green_spaces[col].dropna()

    if series.dtype in ["int64", "float64"]:
        var_type = "Quantitative"
    else:
        n_unique = series.nunique()
        if n_unique == 1:
            var_type = "Constant (single value)"
        elif n_unique == 2:
            var_type = "Binary"
        elif n_unique <= 10:
            var_type = "Categorical — Low cardinality (3–10)"
        elif n_unique <= 100:
            var_type = "Categorical — Medium cardinality (11–100)"
        elif n_unique <= 1000:
            var_type = "Categorical — High cardinality (101–1000)"
        else:
            var_type = "Categorical — Very high cardinality (>1000)"

    categories.append({
        "Column":    col,
        "Dtype":     str(df_green_spaces[col].dtype),
        "Unique":    series.nunique(),
        "Type":      var_type,
        "Missing %": round(df_green_spaces[col].isna().mean() * 100, 1)
    })

var_type_df = pd.DataFrame(categories)
display(var_type_df)

,Column,Dtype,Unique,Type,Missing %
0,nom_projet,object,73,Categorical — Medium cardinality (11–100),0.0
1,arrond,int64,17,Quantitative,0.0
2,secteur_adm,object,16,Categorical — Medium cardinality (11–100),0.0
3,date_fin_t,int64,7,Quantitative,0.0
4,ope_type,object,1,Constant (single value),0.0
5,ind_ev_suppl,float64,60,Quantitative,0.0


#### Verify and filter for valid `arrondissement` values (1-20).

In [7]:
# Filter for valid arrondissements
df_green_spaces = df_green_spaces[(df_green_spaces['arrond'] >= 1) & (df_green_spaces['arrond'] <= 20)]

# Verify values
import numpy as np
check_arrond = np.sort(df_green_spaces['arrond'].unique().astype(int))

check = True
for i in check_arrond:
    if i not in range(1, 21):
        print(f"Invalid arrondissement value: {i}")
        break
        check = False
if check:
    print("All arrondissement values are valid.")

# Show values filtered out
print(f"Invalid records removed: {len(all_results) - len(df_green_spaces)}")
print(f"Records remaining: {len(df_green_spaces)}")

All arrondissement values are valid.
Invalid records removed: 2
Records remaining: 71


#### Renaming Columns French-English

In [8]:
# Tranlsate column names to English

# French-English dictionary
column_translations = {
    "nom_projet": "project_name",
    "arrond": "arrondissement",
    "secteur_adm": "admin_sector",
    "date_fin_t": "completion_date",
    "ope_type": "operation_type",
    "ind_ev_suppl": "added_space_indicator"
}

df_green_spaces = df_green_spaces.rename(columns=column_translations)

In [9]:
# Translate text value of operation_type

df_green_spaces['operation_type'] = df_green_spaces['operation_type'].replace({"Nouvelle surface d’espaces verts": "New green space"})

In [10]:
df_green_spaces.head()

,project_name,arrondissement,admin_sector,completion_date,operation_type,added_space_indicator
0,Transformation de la cour de l’espace Gabriel ...,12,12,2025,New green space,830.0
1,Végétalisation du Centre Sportif Elisabeth,14,14,2023,New green space,243.0
2,Renaturation du square Antoine Blondin,20,20,2022,New green space,949.0
3,Renaturation et extension du square des Périch...,15,15,2021,New green space,860.0
4,Ouverture au public PC18 - Bvd Ornano / rue de...,18,18,2024,New green space,13440.0


## 5. Adding Coordinates via Arrondissement Centroids

The planned green spaces dataset only has an `arrondissement` number, but no coordinates.
To place these projects on a map, we need a latitude and longitude for each one.

We can use the rent control dataset, which contains the polygon shapes of all
80 Paris *quartiers* (neighbourhoods). We will:
1. Load those polygons
2. Merge the 4 quartiers per arrondissement into one shape
3. Compute the centre point (centroid) of each arrondissement
4. Add those coordinates to our planned green spaces dataframe

Every planned green project in the same arrondissement will share the same point, because we do not have a more precise address. This is a known simplification.

#### 5.1  Load the rent control dataset and convert the geometry column. Make sure the generated file "../data/api_rent_control_2025.csv" is uploaded to the notebook session before this step.

In [12]:
import ast
from shapely.geometry import shape
import geopandas as gpd

# Load the rent control CSV
df_rent = pd.read_csv('../data/api_rent_control_2025.csv')

# The 'geographic_shape' column was saved as a string.
# ast.literal_eval converts it back into a Python dictionary.
df_rent['geo_shape'] = df_rent['geo_shape'].apply(ast.literal_eval)

print('Rent control rows loaded:', len(df_rent))
print('Sample geographic_shape type:', type(df_rent['geo_shape'].iloc[0]))

Rent control rows loaded: 320
Sample geographic_shape type: <class 'dict'>


#### 5.2:  Derive the arrondissement number from `postal_code`

In [13]:
# The column 'larger_quarter_code' encodes the arrondissement inside it.
# Example: 7511767:  characters at position 2-4 = '117'  ->  117 - 100 = 17

df_rent['arrond'] = (
    df_rent['postal_code']
    .astype(str)      # make sure it is a string
    .str[3:]         # extract characters at position 4, 5 out of 5 total
    .astype(int)      # convert to integer
)

print('Arrondissements found:', sorted(df_rent['arrond'].unique()))

Arrondissements found: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]


#### 5.3: Extract one polygon per quartier (remove duplicates)

In [14]:
# The rent control CSV has many rows per quartier (one per room type, year, etc.).
# We only need one polygon per quartier, so we keep one row per unique quarter code.

df_quartiers = df_rent.drop_duplicates(subset='quarter_id').copy()

print('Unique quartiers:', len(df_quartiers))  # should be 80 (4 per arrondissement)

Unique quartiers: 80


#### 5.4: Convert the polygon dictionaries into real geometry objects

In [15]:
# Each geographic_shape is a GeoJSON Feature dictionary.
# The polygon coordinates are nested inside ['geometry'].
# shape() from the shapely library converts that into a geometry object that we can work with in Python.

df_quartiers['geometry'] = df_quartiers['geo_shape'].apply(
    lambda x: shape(x['geometry'])
)

print('Sample geometry type:', type(df_quartiers['geometry'].iloc[0]))

Sample geometry type: <class 'shapely.geometry.polygon.Polygon'>


#### 5.5:  Build a GeoDataFrame and dissolve quartiers into arrondissements

In [16]:
# A GeoDataFrame is like a regular DataFrame but with an official geometry column.
# crs='EPSG:4326' means the coordinates are standard lat/lon (WGS84).

gdf_quartiers = gpd.GeoDataFrame(df_quartiers, geometry='geometry', crs='EPSG:4326')

# dissolve() merges all quartier polygons that share the same arrond value
# into one single polygon per arrondissement.
gdf_arrond = gdf_quartiers.dissolve(by='arrond').reset_index()[['arrond', 'geometry']]

print('Arrondissement polygons created:', len(gdf_arrond))  # should be 20

Arrondissement polygons created: 20


#### 5.6: Compute the centroid of each arrondissement

**reproject to EPSG:2154?**  
The coordinates are in degrees (EPSG:4326). Computing a centroid in degrees is
inaccurate, since degrees are not equal-distance units. EPSG:2154 is the official French projection, which uses metres, which gives a geometrically correct centre point.


In [17]:
# Reproject to EPSG:2154 (metres) to get accurate centroids
gdf_projected = gdf_arrond.to_crs('EPSG:2154')

# Compute the centre point of each polygon
gdf_projected['centroid_geom'] = gdf_projected['geometry'].centroid

# Tell GeoPandas to use the centroid column instead of the polygon column,
# then convert the coordinates back to lat/lon (EPSG:4326)
gdf_centroids = gdf_projected.set_geometry('centroid_geom').to_crs('EPSG:4326')

# Extract x (longitude) and y (latitude) from the centroid geometry
gdf_arrond['longitude'] = gdf_centroids.geometry.x.values
gdf_arrond['latitude']  = gdf_centroids.geometry.y.values

# Keep only the three columns we need
df_centroids = gdf_arrond[['arrond', 'latitude', 'longitude']]

print(df_centroids.sort_values('arrond').to_string(index=False))

 arrond  latitude  longitude
      1 48.862563   2.336444
      2 48.868279   2.342803
      3 48.862872   2.360001
      4 48.854341   2.357630
      5 48.844443   2.350715
      6 48.849130   2.332898
      7 48.856175   2.312188
      8 48.872721   2.312554
      9 48.877163   2.337457
     10 48.876130   2.360729
     11 48.859059   2.380059
     12 48.834977   2.421329
     13 48.828388   2.362272
     14 48.829244   2.326542
     15 48.840085   2.292826
     16 48.860392   2.261970
     17 48.887327   2.306775
     18 48.892569   2.348160
     19 48.887075   2.384821
     20 48.863460   2.401189


#### 5.7: Merge the coordinates into the planned green spaces dataframe

In [18]:
# Each planned green space gets the centroid of its arrondissement.
# We match on arrondissement number.

df_green_spaces = df_green_spaces.merge(
    df_centroids,
    left_on  = 'arrondissement',  # column in df_green_spaces
    right_on = 'arrond',           # column in df_centroids
    how      = 'left'              # keep all rows from df_green_spaces
)

# 'arrond' is now a duplicate of 'arrondissement' which we can drop now
df_green_spaces = df_green_spaces.drop(columns='arrond')

# Check: how many rows have no coordinates?
missing = df_green_spaces['latitude'].isna().sum()
print(f'Rows without coordinates: {missing}')
print()
print(df_green_spaces[['project_name', 'arrondissement', 'longitude', 'latitude']].head(10))

Rows without coordinates: 0

                                        project_name  arrondissement  \
0  Transformation de la cour de l’espace Gabriel ...              12   
1         Végétalisation du Centre Sportif Elisabeth              14   
2             Renaturation du square Antoine Blondin              20   
3  Renaturation et extension du square des Périch...              15   
4  Ouverture au public PC18 - Bvd Ornano / rue de...              18   
5  Renaturation et extension du Jardin de la Nouv...               8   
6                                Parking Parc Floral              12   
7           AAP Parisculteurs AUTRES - Tunnel Friant              14   
8                      Renaturation du jardin Curial              19   
9  Requalification de l'avenue du Général Eisenhower               8   

   longitude   latitude  
0   2.421329  48.834977  
1   2.326542  48.829244  
2   2.401189  48.863460  
3   2.292826  48.840085  
4   2.348160  48.892569  
5   2.312554  48.87272

## 4. Exporting Data

In [19]:
# Final check of dataframe
df_green_spaces.head()

,project_name,arrondissement,admin_sector,completion_date,operation_type,added_space_indicator,latitude,longitude
0,Transformation de la cour de l’espace Gabriel ...,12,12,2025,New green space,830.0,48.834977,2.421329
1,Végétalisation du Centre Sportif Elisabeth,14,14,2023,New green space,243.0,48.829244,2.326542
2,Renaturation du square Antoine Blondin,20,20,2022,New green space,949.0,48.863460,2.401189
3,Renaturation et extension du square des Périch...,15,15,2021,New green space,860.0,48.840085,2.292826
4,Ouverture au public PC18 - Bvd Ornano / rue de...,18,18,2024,New green space,13440.0,48.892569,2.348160


In [20]:
# Reset index as project_name
df_green_spaces = df_green_spaces.set_index('project_name')

# Check dataframe
df_green_spaces.head()

,arrondissement,admin_sector,completion_date,operation_type,added_space_indicator,latitude,longitude
project_name,,,,,,,
Transformation de la cour de l’espace Gabriel Lamé en jardin public,12,12,2025,New green space,830.0,48.834977,2.421329
Végétalisation du Centre Sportif Elisabeth,14,14,2023,New green space,243.0,48.829244,2.326542
Renaturation du square Antoine Blondin,20,20,2022,New green space,949.0,48.863460,2.401189
Renaturation et extension du square des Périchaux (phase 2),15,15,2021,New green space,860.0,48.840085,2.292826
Ouverture au public PC18 - Bvd Ornano / rue des Poissonniers,18,18,2024,New green space,13440.0,48.892569,2.348160


In [21]:
# Checking for valid arrondissement values
df_green_spaces['arrondissement'].unique()

array([12, 14, 20, 15, 18,  8, 19, 16, 10,  4, 13,  2, 11, 17,  7,  5])

In [22]:
# Save dataset as CSV
df_green_spaces.to_csv("../data/planned_green_spaces.csv")
df_green_spaces.shape

(71, 7)